# 🛠️ Preparación del entorno · Taller UEX26

Este notebook prepara el entorno que utilizaremos en los talleres:

- **GeoBrief · CrewAI + Ollama**
- **Neo4j AuraDB + OpenAI + LangChain**

Al ejecutarlo se creará un entorno **Conda** llamado **`uex26`**, se instalarán las dependencias de Python necesarias y se registrará un kernel de Jupyter llamado **`Python (uex26)`**.

> **Importante:** este notebook se ejecuta una sola vez, desde cualquier kernel que ya tenga acceso a Conda.  
> Cuando termine, cambia el kernel de los notebooks del taller a **`Python (uex26)`**.

### Requisito previo

Debes tener instalada una distribución que incluya Conda, por ejemplo **Miniconda** o **Anaconda**.

El taller de GeoBrief utiliza además **Ollama**, que es una aplicación externa y no se instala con `pip`. Al final del notebook encontrarás una comprobación específica.


## 1) Comprobar que Conda está disponible

La siguiente celda localiza el ejecutable de Conda y muestra su versión.


In [ ]:
import shutil
import subprocess
from pathlib import Path

conda_exe = shutil.which("conda")

if conda_exe is None:
    # Rutas habituales cuando Jupyter no hereda el PATH completo.
    candidates = [
        Path.home() / "miniconda3" / "bin" / "conda",
        Path.home() / "anaconda3" / "bin" / "conda",
        Path.home() / "miniforge3" / "bin" / "conda",
        Path.home() / "mambaforge" / "bin" / "conda",
    ]
    conda_exe = next((str(p) for p in candidates if p.exists()), None)

if conda_exe is None:
    raise RuntimeError(
        "No se ha encontrado Conda. Instala Miniconda/Anaconda y vuelve a ejecutar este notebook."
    )

print("Conda encontrado en:", conda_exe)
subprocess.run([conda_exe, "--version"], check=True)


## 2) Crear el entorno `uex26`

Usaremos **Python 3.11**, que ofrece una buena compatibilidad con las librerías utilizadas en ambos talleres.

La celda es segura para volver a ejecutarse: si el entorno ya existe, no intenta crearlo de nuevo.


In [ ]:
import json
import subprocess

ENV_NAME = "uex26"
PYTHON_VERSION = "3.11"

result = subprocess.run(
    [conda_exe, "env", "list", "--json"],
    check=True,
    capture_output=True,
    text=True,
)

envs = json.loads(result.stdout).get("envs", [])
env_names = {Path(env).name for env in envs}

if ENV_NAME in env_names:
    print(f"✅ El entorno '{ENV_NAME}' ya existe.")
else:
    print(f"Creando el entorno '{ENV_NAME}' con Python {PYTHON_VERSION}...")
    subprocess.run(
        [
            conda_exe,
            "create",
            "-n", ENV_NAME,
            f"python={PYTHON_VERSION}",
            "pip",
            "-y",
        ],
        check=True,
    )
    print(f"✅ Entorno '{ENV_NAME}' creado correctamente.")


## 3) Instalar las dependencias del taller

Se instalarán conjuntamente las librerías utilizadas por los dos notebooks.

### GeoBrief · CrewAI + Ollama

- `crewai[litellm]`
- `requests`

### Neo4j AuraDB + OpenAI + LangChain

- `python-dotenv`
- `neo4j`
- `langchain`
- `langchain-community`
- `langchain-openai`
- `langchain-neo4j`
- `tiktoken`

### Jupyter

- `jupyterlab`
- `ipykernel`

La instalación se realiza **dentro de `uex26`**, aunque este notebook esté ejecutándose desde otro kernel.


In [ ]:
import subprocess

PACKAGES = [
    # Jupyter
    "jupyterlab",
    "ipykernel",

    # GeoBrief
    "crewai[litellm]",
    "requests",

    # AuraDB + OpenAI + LangChain
    "python-dotenv",
    "neo4j",
    "langchain",
    "langchain-community",
    "langchain-openai",
    "langchain-neo4j",
    "tiktoken",
]

print("Actualizando herramientas básicas de instalación...")
subprocess.run(
    [
        conda_exe, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        "--upgrade", "pip", "setuptools", "wheel"
    ],
    check=True,
)

print("\nInstalando dependencias del taller...")
subprocess.run(
    [
        conda_exe, "run", "-n", ENV_NAME,
        "python", "-m", "pip", "install",
        *PACKAGES
    ],
    check=True,
)

print("\n✅ Dependencias instaladas correctamente.")


## 4) Registrar `uex26` como kernel de Jupyter

Esto permite seleccionar el entorno directamente desde JupyterLab o VS Code.

El nombre que aparecerá en el selector de kernels será:

**Python (uex26)**


In [ ]:
import subprocess

subprocess.run(
    [
        conda_exe, "run", "-n", ENV_NAME,
        "python", "-m", "ipykernel", "install",
        "--user",
        "--name", ENV_NAME,
        "--display-name", "Python (uex26)",
    ],
    check=True,
)

print("✅ Kernel registrado como 'Python (uex26)'.")


## 5) Verificar la instalación

La siguiente celda ejecuta Python **dentro de `uex26`** e intenta importar las librerías principales.

Si todas aparecen con `OK`, el entorno está preparado.


In [ ]:
import subprocess
import textwrap

verification_code = r"""
import sys
import importlib

modules = {
    "requests": "requests",
    "dotenv": "python-dotenv",
    "neo4j": "neo4j",
    "langchain": "langchain",
    "langchain_community": "langchain-community",
    "langchain_openai": "langchain-openai",
    "langchain_neo4j": "langchain-neo4j",
    "tiktoken": "tiktoken",
    "crewai": "crewai",
    "litellm": "litellm",
    "IPython": "IPython",
    "ipykernel": "ipykernel",
}

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
print()

errors = []

for module_name, package_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "versión no expuesta")
        print(f"OK  {package_name:<22} {version}")
    except Exception as exc:
        errors.append((package_name, str(exc)))
        print(f"ERROR  {package_name}: {exc}")

if errors:
    raise SystemExit(
        "\nLa verificación ha encontrado dependencias con problemas: "
        + ", ".join(name for name, _ in errors)
    )

print("\n✅ Entorno Python preparado para los dos talleres.")
"""

subprocess.run(
    [
        conda_exe, "run", "-n", ENV_NAME,
        "python", "-c", verification_code
    ],
    check=True,
)


## 6) Comprobar Ollama

**Ollama no es una librería de Python**, por lo que no forma parte del entorno Conda.

Para el taller **GeoBrief · CrewAI + Ollama** necesitas:

1. Tener Ollama instalado en el ordenador.
2. Tener disponible el modelo `llama3.1:latest`.
3. Tener Ollama ejecutándose cuando uses el notebook.

La siguiente celda comprueba si el comando `ollama` está disponible. No descarga automáticamente el modelo, ya que puede ocupar varios GB.


In [ ]:
import shutil
import subprocess

ollama_exe = shutil.which("ollama")

if ollama_exe is None:
    print("⚠️ Ollama no está disponible en el PATH.")
    print("Instálalo desde https://ollama.com/download antes del taller de GeoBrief.")
else:
    print("✅ Ollama encontrado en:", ollama_exe)
    subprocess.run([ollama_exe, "--version"], check=False)

    result = subprocess.run(
        [ollama_exe, "list"],
        capture_output=True,
        text=True,
        check=False,
    )

    print("\nModelos instalados:")
    print(result.stdout.strip() or "(ninguno)")

    if "llama3.1" not in result.stdout:
        print("\n⚠️ Falta Llama 3.1.")
        print("Ejecuta en una terminal:")
        print("ollama pull llama3.1:latest")
    else:
        print("\n✅ Llama 3.1 está disponible.")


## 7) Finalizar la preparación

Si las comprobaciones anteriores han terminado correctamente:

1. **Reinicia Jupyter o VS Code** si el nuevo kernel no aparece inmediatamente.
2. Abre el notebook del taller.
3. Selecciona el kernel **`Python (uex26)`**.
4. Comprueba que la versión de Python pertenece al entorno correcto.

Puedes verificarlo dentro de cualquier notebook con:

```python
import sys
print(sys.executable)
```

La ruta mostrada debería contener `uex26`.

---

### Activación manual desde una terminal

También puedes utilizar el entorno fuera de Jupyter:

```bash
conda activate uex26
```

Para iniciar JupyterLab desde ese entorno:

```bash
conda activate uex26
jupyter lab
```

> No es necesario volver a crear el entorno cada vez. Una vez preparado, basta con activarlo o seleccionar su kernel.
